# Cell 1 — Markdown
# Step 6: Baseline Model Development

## Objective

The objective of this step is to build baseline machine learning models that predict whether a customer will respond to a commercial campaign.

The target variable is:

**y = customer campaign response**

In the UCI Bank Marketing dataset:

- `yes` means the customer subscribed to the term deposit
- `no` means the customer did not subscribe

This step compares three baseline models:

| Model | Purpose |
|---|---|
| Logistic Regression | Interpretable baseline model |
| Random Forest | Nonlinear model that captures complex patterns |
| Gradient Boosting | Strong predictive model for campaign response prediction |

The final recommended model will be selected based on predictive performance and business usefulness.

In [4]:
# Cell 2 — Import Libraries
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [5]:
# Cell 3 — Locate Project Folder and Load Data
# Find project root folder
current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Could not find the data folder. Make sure you are working inside the project repository.")

# Dataset path
data_path = PROJECT_ROOT / "data" / "raw" / "bank-additional-full.csv"

# Load dataset
df = pd.read_csv(data_path, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

FileNotFoundError: Could not find the data folder. Make sure you are working inside the project repository.

In [ ]:
# Cell 4 — Prepare Target Variable
# Check target values
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True) * 100)

In [ ]:
# Cell 4 — Prepare Target Variable
# Convert target variable to binary format
# yes = 1 means customer responded
# no = 0 means customer did not respond

df["target"] = df["y"].map({"yes": 1, "no": 0})

# Drop original target column
df_model = df.drop(columns=["y"])

df_model.head()

In [ ]:
# Cell 5 — Define Features and Target
X = df_model.drop(columns=["target"])
y = df_model["target"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Cell 6 — Identify Numerical and Categorical Variables
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
# Cell 7 — Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining response rate:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest response rate:")
print(y_test.value_counts(normalize=True) * 100)


In [ ]:
# Cell 8 — Build Preprocessing Pipeline
# Version-compatible OneHotEncoder
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder)
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

In [ ]:
# Cell 9 — Define Baseline Models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.10,
        max_depth=3,
        random_state=42
    )
}

models

In [ ]:
# Cell 10 — Train and Evaluate Models
results = []

trained_models = {}

for model_name, model in models.items():
    print(f"Training model: {model_name}")
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_prob),
        "PR AUC": average_precision_score(y_test, y_prob)
    }
    
    results.append(metrics)
    trained_models[model_name] = pipeline

print("Model training completed.")

In [ ]:
# Cell 11 — Compare Model Results
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["ROC AUC", "PR AUC"],
    ascending=False
)

results_df

In [ ]:
# Cell 12 — Save Model Comparison Results
reports_path = PROJECT_ROOT / "reports"
reports_path.mkdir(exist_ok=True)

results_file = reports_path / "baseline_model_results.csv"

results_df.to_csv(results_file, index=False)

print(f"Baseline model results saved to: {results_file}")

In [ ]:
# Cell 13 — Select Best Model
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best baseline model:")
print(best_model_name)

results_df.iloc[0]

In [ ]:
# Cell 14 — Detailed Evaluation of Best Model
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred_best))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))


#Cell 15 — Markdown Interpretation
## Baseline Model Results Summary

Three baseline models were trained and evaluated:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting

Because the campaign-response problem is imbalanced, accuracy alone is not enough to evaluate model quality. The most important metrics are:

- **ROC AUC**, because it measures the model's ability to rank responders above non-responders
- **PR AUC**, because it is useful when the positive response class is relatively small
- **Recall**, because the business wants to identify as many likely responders as possible
- **Precision**, because outreach resources should not be wasted on customers unlikely to respond

The best baseline model should be selected based on ranking ability and commercial usefulness, not only raw accuracy.

For this project, the recommended final model candidate is the model with the strongest ROC AUC and PR AUC performance. In most campaign-response problems, either **Gradient Boosting** or **Random Forest** is expected to perform better than Logistic Regression because these models can capture nonlinear customer behavior patterns.

In [ ]:
# Cell 16 — Save Customer-Level Prediction Scores
scored_customers = X_test.copy()

scored_customers["actual_response"] = y_test.values
scored_customers["predicted_response_probability"] = y_prob_best
scored_customers["predicted_response"] = y_pred_best

scored_customers = scored_customers.sort_values(
    by="predicted_response_probability",
    ascending=False
)

processed_path = PROJECT_ROOT / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

scores_file = processed_path / "baseline_customer_response_scores.csv"

scored_customers.to_csv(scores_file, index=False)

print(f"Customer response scores saved to: {scores_file}")

scored_customers.head(10)